### Note: The following notebook has been used to generate the plots for the website on RQ3 part. Since it had a lot of trial and error, it has a file on his own. The code is identical to that of the `results.ipyb` untill the genereated plot.

In [ ]:
from config import *
import pandas as pd
from src.website_plots.RQ3_plots import *

from src.models.log_log_efficiency_model import linear_log_log_model
from src.utils.data_utils import add_structural_factors, split_channels_by_subscriber_tier_weighted


In [2]:
channel_df = pd.read_parquet(DATASET_CHANNELS)
channel_df.head(6)

,category_cc,join_date,channel,name_cc,subscribers_cc,videos_cc,subscriber_rank_sb,weights,channel_age_days,videos_per_day,avg_engagement_efficiency,total_views,total_duration,total_like_count,total_dislike_count,total_num_comms,avg_video_like_ratio,video_count_processed
0,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087,3468.0,1.140715,0.041988,2.345148e+10,3449869.0,950538161.0,24615813.0,72904068.0,0.982724,3927
1,Education,2006-09-01,UCbCmjCuTUZos6Inko4u57UQ,Cocomelon - Nursery ...,60100000,458,7.0,2.087,4804.0,0.095337,0.002954,4.323417e+10,519007.0,96951076.0,55216214.0,611177.0,0.633412,462
2,Entertainment,2006-09-20,UCpEhnqL0y41EpW2TvWAHD7Q,SET India,56018869,32661,8.0,2.087,4785.0,6.825705,0.004994,4.075935e+10,30501034.0,139097733.0,21976958.0,5557559.0,0.861188,23903
3,Howto & Style,2016-11-15,UC295-Dw_tDNtZXFeAPAW6Aw,5-Minute Crafts,60600000,3591,9.0,2.087,1076.0,3.337361,0.011340,1.596080e+10,3383229.0,109974549.0,13606395.0,4506589.0,0.904908,3624
4,Sports,2007-05-11,UCJ5v_MCY6GNUBTO8-D3XoAg,WWE,48400000,43421,11.0,2.087,4552.0,9.538884,0.015842,3.624948e+10,7868933.0,330247613.0,18966387.0,19107723.0,0.939069,40549
5,Entertainment,2007-01-15,UCIwFjwMjI0y7PDBVEO9-bkQ,Justin Bieber,46574085,134,12.0,2.087,4668.0,0.028706,0.025269,5.886436e+08,17697.0,8815774.0,373636.0,719646.0,0.957442,128


In [3]:
timeseries_df = pd.read_parquet(DATASET_TIMESERIES)
timeseries_df.head(5)

,channel,category,datetime,views,delta_views,subs,delta_subs,videos,delta_videos,activity,week_start,efficiency_4week,efficiency_4week_filled,posting_pace_4w,posting_consistency_4w,posting_pace_8w,posting_consistency_8w,posting_pace_12w,posting_consistency_12w
0,UC--24Q3_ZQeFmgJE-Um5QZQ,Travel & Events,2016-08-01,62524.500,4113.500,1466.000,0.000,29,0,1,2016-07-26,0.049820,0.049820,0.0,0.000000,0.0,0.000000,0.0,0.000000
1,UC--24Q3_ZQeFmgJE-Um5QZQ,Travel & Events,2016-08-08,67600.250,5075.750,1508.625,42.625,30,1,2,2016-08-02,0.049374,0.049374,0.0,1.000000,0.0,1.000000,0.0,1.000000
2,UC--24Q3_ZQeFmgJE-Um5QZQ,Travel & Events,2016-08-15,69982.000,2381.750,1530.750,22.125,31,1,2,2016-08-09,0.045413,0.045413,0.0,0.000000,0.0,0.000000,0.0,0.000000
3,UC--24Q3_ZQeFmgJE-Um5QZQ,Travel & Events,2016-08-22,72663.125,2681.125,1558.625,27.875,32,1,2,2016-08-16,0.046415,0.046415,1.0,0.000000,1.0,0.000000,1.0,0.000000
4,UC--24Q3_ZQeFmgJE-Um5QZQ,Travel & Events,2016-08-29,75093.000,2429.875,1591.000,32.375,33,1,2,2016-08-23,0.031169,0.031169,1.0,0.333333,1.0,0.087129,1.0,0.087129


In [4]:
# Add residuals from Log-Log Model to Channels DataFrame
channels_with_residuals = linear_log_log_model(channel_df, "subscribers_cc", "avg_engagement_efficiency", weight_col="weights", verbose=False)

# Aggregate (median) Structural Factors from Timeseries (pace, consistency) and Videos (duration) per Channel
channels_with_residuals = add_structural_factors(channels_with_residuals, timeseries_df, videos_dataset=DATASET_VIDEO_LIGHT)

# split into Small and Mid-Sized Channel as defined before
small_df, mid_df = split_channels_by_subscriber_tier_weighted(channels_with_residuals)

In [5]:


heatmap = plot_recipe_maps_3x2(
    small_df,
    mid_df,
    viridis_wash=0.15
)
heatmap.show()
# save as html
heatmap.write_html("plots/recipe_maps_3x2.html")

In [6]:
corr = plot_corr_small_mid(
    small_df,
    mid_df,
    pace_col="pace8_median",
    cons_col="cons8_median",
    viridis_wash=0.15
)

corr.show()
# save as html
corr.write_html("plots/corr_small_mid.html")

In [7]:
from src.models.spec_model import run_spec_small_mid

In [8]:
small_df.columns

Index(['category_cc', 'join_date', 'channel', 'name_cc', 'subscribers_cc',
       'videos_cc', 'subscriber_rank_sb', 'weights', 'channel_age_days',
       'videos_per_day', 'avg_engagement_efficiency', 'total_views',
       'total_duration', 'total_like_count', 'total_dislike_count',
       'total_num_comms', 'avg_video_like_ratio', 'video_count_processed',
       'log_x', 'log_y', 'pred_log_y', 'residual', 'zscore', 'pace8_median',
       'cons8_median', 'pace4_median', 'cons4_median', 'pace12_median',
       'cons12_median', 'video_count_from_pace', 'duration_median',
       'log_duration'],
      dtype='object')

In [9]:
models_4, effects_4, coefs_4 = run_spec_small_mid(
    small_df, mid_df,
    pace_col="pace4_median",
    cons_col="cons4_median",
)

models_8, effects_8, coefs_8 = run_spec_small_mid(
    small_df, mid_df,
    pace_col="pace8_median",
    cons_col="cons8_median",
)

models_12, effects_12, coefs_12 = run_spec_small_mid(
    small_df, mid_df,
    pace_col="pace12_median",
    cons_col="cons12_median",
)

In [10]:
effects_12

,tier,spec,var,p25,p75,span,%ΔE_p25->p75,%ΔE_CI_low,%ΔE_CI_high
0,small,pace12_median+cons12_median,pace12_median,0.000000,1.000000,1.000000,-0.372704,-0.655582,-0.089021
1,small,pace12_median+cons12_median,cons12_median,0.000000,0.438260,0.438260,18.071795,14.373706,21.889456
2,small,pace12_median+cons12_median,log_duration,5.164786,6.270043,1.105257,29.056812,23.895441,34.433202
3,mid,pace12_median+cons12_median,pace12_median,0.000000,1.000000,1.000000,-0.288727,-0.369973,-0.207415
4,mid,pace12_median+cons12_median,cons12_median,0.000000,0.502798,0.502798,18.894106,15.046763,22.870110
5,mid,pace12_median+cons12_median,log_duration,5.249652,6.364751,1.115099,31.091369,26.105025,36.274879


In [11]:
coefs_4_fig = plot_coef_bars(coefs_4, var_order=("log_duration", "cons4_median", "pace4_median"))
effects_4_fig = plot_effects_bars(effects_4, var_order=("log_duration", "cons4_median", "pace4_median"))

coefs_8_fig = plot_coef_bars(coefs_8, var_order=("log_duration", "cons8_median", "pace8_median"))
effects_8_fig = plot_effects_bars(effects_8, var_order=("log_duration", "cons8_median", "pace8_median"))

coefs_12_fig = plot_coef_bars(coefs_12, var_order=("log_duration", "cons12_median", "pace12_median"))
effects_12_fig = plot_effects_bars(effects_12, var_order=("log_duration", "cons12_median", "pace12_median"))

In [12]:
coefs_4_fig.show()
effects_4_fig.show()
coefs_8_fig.show()
effects_8_fig.show()
coefs_12_fig.show()
effects_12_fig.show()

In [13]:
# save plots as html
coefs_4_fig.write_html("plots/coefs_4.html")
effects_4_fig.write_html("plots/effects_4.html")
coefs_8_fig.write_html("plots/coefs_8.html")
effects_8_fig.write_html("plots/effects_8.html")
coefs_12_fig.write_html("plots/coefs_12.html")
effects_12_fig.write_html("plots/effects_12.html")
